[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# Loading and N+1


## What you will be able to do

Count the statements a loop sends, rather than guess at them, and recognize the shape that sends one
for every row it read. Load a relationship on purpose with `selectinload` and `joinedload`, know
which of the two to reach for, and read what each one sends. Set a loading strategy on the model
itself and say why that is usually the wrong place for it. Recognize the failures: the loop that is
twenty-one queries, the attribute that cannot be read once the session has closed, and the model
that loads half the database on every query about it.


## The idea

### The problem

A page lists twenty teams with the number of heroes on each. The code for it is four lines, reads
well, and is correct. It sends twenty-one queries: one for the teams, and one more for every team as
the loop reaches it.

Nothing about that is visible from the code. `team.heroes` looks like an attribute, the loop looks
like a loop, and the queries are sent one at a time by the thing that looks least like a query. On a
database in memory with twenty rows nobody notices. On a server with a network in between, twenty-one
round trips is the difference between a page that loads and a page that does not, and the number
grows with the list.

This has a name, N+1, and it is the most common performance problem in anything built on an ORM. It
is also easy to fix once it can be seen, which is why this notebook counts statements rather than
timing them: a count is the same on every machine and says exactly what happened.

### What a loading strategy is

> A **loading strategy** says when a relationship is fetched. The default is **lazy**: nothing is
> fetched until the attribute is read, and then one query goes out for it.
> **`selectinload(Team.heroes)`**, passed to **`.options(...)`** on a statement, fetches the
> relationship for every row of that query in one more statement, with an `IN` list of the ids.
> **`joinedload(Hero.team)`** fetches it in the same statement, with a join. Both are per query.
> The same choice can be made on the class instead, with
> **`sa_relationship_kwargs={"lazy": "selectin"}`**, which applies to every query that loads it.

### Why it works that way

- **A lazy load is one query per attribute read.** The session keeps what it has already loaded, so
  the count follows the number of **distinct** objects on the other side, not the number of reads.
- **`selectinload` is a second query for the whole page.** One `SELECT ... WHERE id IN (...)`, which
  scales with the size of the page rather than with its rows, and which keeps the rows of the first
  query as they were.
- **`joinedload` is one query with a join.** No second round trip, and every column of the other
  side comes back beside every row of this one, which for a list of children means the parent's
  columns repeated for each child.
- **The strategy is a property of the query, not of the model.** The same relationship is wanted on
  one page and not on the next, which is why the option goes on the statement.
- **It comes from `sqlalchemy.orm`.** SQLModel does not re-export `selectinload`, and this is one of
  the places where the answer is SQLAlchemy's, as `sa_column` is in the
  **sa_column and __table_args__** notebook.

### Where this shows up

Every list page in every application, and every report that walks a relationship. The
**SQLAlchemy, Deep Dive** guide's Loading Strategies notebook is the long version, with
`lazy="raise"`, `contains_eager` and the rest. The **SQLModel in FastAPI** notebook is where this
one lands in practice: a route that returns a list of teams with their heroes needs the option, and
the session closes at the end of the request, which turns a missed one into an error rather than a
slow page.

### What this notebook covers

- Counting what a loop sends
- Why it is twenty-one, and what the number follows
- `selectinload`: one more query for the whole page
- `joinedload`: no more queries at all
- Which to reach for
- The strategy on the model, and why not
- Loading for what happens after the session closes
- A roster in two queries, finished
- Three failures, from a loop of twenty-one queries to a model that loads everything

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import event
from sqlalchemy.orm import selectinload
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str
    team_id: int | None = Field(default=None, foreign_key="team.id")
    team: Team | None = Relationship(back_populates="heroes")


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
sent = []
event.listen(engine, "before_cursor_execute", lambda *arguments: sent.append(arguments[2]))

with Session(engine) as session:
    for number in range(3):
        session.add(Team(name=f"Team {number}", heroes=[Hero(name=f"Hero {number}")]))
    session.commit()

    sent.clear()
    for team in session.exec(select(Team)).all():
        len(team.heroes)
    print("one at a time:", len(sent), "statements")

    session.expire_all()
    sent.clear()
    for team in session.exec(select(Team).options(selectinload(Team.heroes))).all():
        len(team.heroes)
    print("loaded on purpose:", len(sent), "statements")
```

```
one at a time: 4 statements
loaded on purpose: 2 statements
```

Three teams, and the same loop twice. The first sent one query for the teams and one for each team's
heroes as the loop reached it. The second asked for the heroes to be loaded with the teams, and sent
two statements however many teams there are.


## Setup

Fourteen imports, one of them installed first where it is missing, four helpers, the classes, the
engine, and a database with rather more in it than the earlier notebooks.

- `sqlmodel` is the library, and `SQLModel`, `Field`, `Relationship`, `Session`, `create_engine` and
  `select`, from it, are the classes, the lists, the session and the reading. Colab does not have
  SQLModel, so the cell installs 0.0.42 with `pip` where it is missing, and `version` and
  `PackageNotFoundError`, from `importlib.metadata`, `subprocess` and `sys` find out whether it is
- `selectinload` and `joinedload`, from `sqlalchemy.orm`, are the two loading strategies this
  notebook uses, and `DetachedInstanceError`, from `sqlalchemy.orm.exc`, is what a relationship read
  after its session closed raises
- `event` and `insert`, from `sqlalchemy`, are the pragma on every connection, the rows loaded
  without a session, and the hook `counting` listens on
- `contextlib` and `Counter`, from `collections`, make `counting`, which counts the statements an
  engine sends inside a block. A count is the measurement here rather than a time, because a time
  differs on every run and on every machine while a count does not
- `logging` carries the SQL an engine logs to `PrintStatements`, which is how the two examples that
  show the statements themselves print them without a time in front of every line
- `re` takes memory addresses out of a message, `Path` names the database file, and `shutil` removes
  the scratch folder at the start and at the end
- `subprocess` and `sys` also run `run_python`, for the strategy that has to be set on a class

The cast is not the eight heroes this time. `build_many` writes twenty teams and two hundred heroes,
ten to a team, from two lists and a formula, so that the counts below are the same on every machine.


In [1]:
import contextlib
import logging
import re
import shutil
import subprocess
import sys
from collections import Counter
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import sqlmodel
from sqlalchemy import event, func, insert
from sqlalchemy.orm import joinedload, selectinload
from sqlalchemy.orm.exc import DetachedInstanceError
from sqlmodel import Field, Relationship, Session, SQLModel, col, create_engine, select

def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


@contextlib.contextmanager
def counting(engine):
    """The statements an engine sends inside the block, counted by their first word."""
    counted = Counter()

    def count(connection, cursor, statement, parameters, context, executemany):
        counted[statement.split()[0].upper()] += 1

    event.listen(engine, "before_cursor_execute", count)
    try:
        yield counted
    finally:
        event.remove(engine, "before_cursor_execute", count)


class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again

def run_python(path):
    """Run a file in a Python of its own and print what it printed."""
    done = subprocess.run([sys.executable, path], capture_output=True, text=True)
    print(done.stdout.strip() or done.stderr.strip().splitlines()[-1])


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)

    heroes: list["Hero"] = Relationship(back_populates="team")


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


def hero_engine(path=None, echo=False):
    """The guide's engine: the database in a file, or with no path one in memory, with foreign keys checked."""
    engine = create_engine("sqlite://" if path is None else f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(connection, record):
        cursor = connection.cursor()
        cursor.execute("PRAGMA foreign_keys=ON")
        cursor.close()

    return engine


PREFIXES = ["Iron", "Silver", "Night", "Storm", "Ghost", "Solar", "Crimson", "Jade", "Cobalt", "Ember"]
BEASTS = ["Fox", "Hawk", "Wolf", "Lark", "Viper", "Moth", "Crane", "Otter", "Falcon", "Bear",
          "Lynx", "Heron", "Stag", "Raven", "Mole", "Owl", "Pike", "Shrike", "Boar", "Kite"]

BIG_TEAMS = [f"{prefix} Squad" for prefix in PREFIXES] + [f"{beast} Watch" for beast in BEASTS[:10]]
BIG_HEROES = [f"{prefix}-{beast}" for prefix in PREFIXES for beast in BEASTS]


def build_many(engine):
    """Twenty teams and two hundred heroes, ten to a team, from the lists above and no random."""
    SQLModel.metadata.create_all(engine)
    with engine.begin() as connection:
        connection.execute(insert(Team), [{"name": name, "headquarters": f"{name} House"}
                                          for name in BIG_TEAMS])
        connection.execute(insert(Hero), [{"name": name, "secret_name": f"Recruit {number:03d}",
                                           "age": 20 + number % 40, "team_id": (number % 20) + 1}
                                          for number, name in enumerate(BIG_HEROES)])

shutil.rmtree("scratch", ignore_errors=True)                        # a rerun starts from the same rows
Path("scratch").mkdir()
engine = hero_engine("scratch/heroes.db")
build_many(engine)

with Session(engine) as session:
    print("sqlmodel", sqlmodel.__version__, "|", len(session.exec(select(Hero)).all()), "heroes in",
          len(session.exec(select(Team)).all()), "teams")


sqlmodel 0.0.42 | 200 heroes in 20 teams


## Worked examples

### Counting what a loop sends

A page of teams, each with the number of heroes on it:


In [2]:
with Session(engine) as session, counting(engine) as sent:
    rows = [(team.name, len(team.heroes)) for team in session.exec(select(Team)).all()]

print("the first three rows:", rows[:3])
print("statements sent     :", dict(sent))


the first three rows: [('Iron Squad', 10), ('Silver Squad', 10), ('Night Squad', 10)]
statements sent     : {'SELECT': 21}


Twenty-one statements for twenty rows. One `SELECT` for the teams, and then one more each time the
loop read a team's heroes, sent at that moment and not before. The code says nothing about it: the
only thing in that line that touches a database is the word `heroes`.

### Why it is twenty-one, and what the number follows

The other direction looks different and counts the same:


In [3]:
with Session(engine) as session, counting(engine) as sent:
    for hero in session.exec(select(Hero)).all():
        hero.team.name
print("two hundred heroes, reading the team:", dict(sent))

with Session(engine) as session, counting(engine) as sent:
    for hero in session.exec(select(Hero).limit(50)).all():
        hero.team.name
print("the same over fifty heroes          :", dict(sent))


two hundred heroes, reading the team: {'SELECT': 21}
the same over fifty heroes          : {'SELECT': 21}


Two hundred reads, twenty-one statements, and fifty reads, twenty-one statements. The count follows
the number of **distinct** teams, not the number of heroes: once a team has been loaded the session
has it, and reading it again is free. That is worth knowing, because it is why the problem is so
often missed in testing, where the same few parents come back every time, and so obvious in
production, where a page of a hundred orders has a hundred different customers on it.

### selectinload: one more query for the whole page

`options` puts the strategy on the statement:


In [4]:
engine.echo = True
with Session(engine) as session, counting(engine) as sent:
    teams = session.exec(select(Team).options(selectinload(Team.heroes)).limit(3)).all()
    rows = [(team.name, len(team.heroes)) for team in teams]
engine.echo = False
print("statements:", dict(sent), "| rows:", rows)


    BEGIN (implicit)
    SELECT team.id, team.name, team.headquarters
    FROM team
     LIMIT ? OFFSET ?
    values: (3, 0)
    SELECT hero.team_id AS hero_team_id, hero.id AS hero_id, hero.name AS hero_name, hero.secret_name AS hero_secret_name, hero.age AS hero_age
    FROM hero
    WHERE hero.team_id IN (?, ?, ?)
    values: (1, 2, 3)
    ROLLBACK
statements: {'SELECT': 2} | rows: [('Iron Squad', 10), ('Silver Squad', 10), ('Night Squad', 10)]


Two statements. The first is the teams as before; the second fetches every hero whose `team_id` is
in the list of teams the first one found, which is the `IN` in the second statement. The loop then
reads what is already in memory and sends nothing at all.

It is two statements for three teams and two for two hundred, which is the point: the cost stops
growing with the size of the page.

### joinedload: no more queries at all

`joinedload` asks for the other side in the same statement:


In [5]:
engine.echo = True
with Session(engine) as session, counting(engine) as sent:
    heroes = session.exec(select(Hero).options(joinedload(Hero.team)).limit(3)).all()
    rows = [(hero.name, hero.team.name) for hero in heroes]
engine.echo = False
print("statements:", dict(sent), "| rows:", rows)


    BEGIN (implicit)
    SELECT hero.id, hero.name, hero.secret_name, hero.age, hero.team_id, team_1.id AS id_1, team_1.name AS name_1, team_1.headquarters
    FROM hero LEFT OUTER JOIN team AS team_1 ON team_1.id = hero.team_id
     LIMIT ? OFFSET ?
    values: (3, 0)
    ROLLBACK
statements: {'SELECT': 1} | rows: [('Iron-Fox', 'Iron Squad'), ('Iron-Hawk', 'Silver Squad'), ('Iron-Wolf', 'Night Squad')]


One statement, with a `LEFT OUTER JOIN`, and every team column beside every hero row. For a
many-to-one like `Hero.team` that is exactly right: one round trip, and a team repeated across the
heroes that share it costs almost nothing.

For a one-to-many it is a different trade. `joinedload(Team.heroes)` returns one row per hero with
the team's columns repeated on each, and with `limit` it needs a subquery to keep the limit meaning
teams rather than rows, which SQLAlchemy writes for you and which is more work than the second
statement `selectinload` sends.

### Which to reach for

| The relationship | Reach for | Why |
|---|---|---|
| many to one, such as `Hero.team` | `joinedload` | one statement, and the other side is one row |
| one to many, such as `Team.heroes` | `selectinload` | one extra statement, and no rows repeated |
| many to many, through a link | `selectinload` | the same, with the link table joined in the second |
| more than one level deep | `selectinload(...).selectinload(...)` | one statement for each level, not one per row |
| you are not sure | `selectinload` | it is never much worse, and it never repeats rows |

### The strategy on the model, and why not

A relationship can carry its own strategy, and then every query that loads that model uses it:


In [6]:
%%writefile scratch/eager_model.py
import contextlib
from collections import Counter

from sqlalchemy import event
from sqlmodel import Field, Relationship, Session, SQLModel, create_engine, select


class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)

    heroes: list["Hero"] = Relationship(back_populates="team",
                                        sa_relationship_kwargs={"lazy": "selectin"})


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=50)
    team_id: int | None = Field(default=None, foreign_key="team.id")

    team: Team | None = Relationship(back_populates="heroes")


engine = create_engine("sqlite://")
SQLModel.metadata.create_all(engine)
with Session(engine) as session:
    for number in range(20):
        session.add(Team(name=f"Team {number}", heroes=[Hero(name=f"Hero {number}")]))
    session.commit()

sent = Counter()
event.listen(engine, "before_cursor_execute",
             lambda connection, cursor, statement, *rest: sent.update([statement.split()[0].upper()]))

with Session(engine) as session:
    names = [team.name for team in session.exec(select(Team)).all()]    # the heroes are not wanted
print("a query that needs only names:", dict(sent), "and it loaded", len(names), "teams")


Writing scratch/eager_model.py


In [7]:
run_python("scratch/eager_model.py")


a query that needs only names: {'SELECT': 2} and it loaded 20 teams


Two statements for a query that wanted one column. Nothing in that code asked for the heroes, and
they were fetched anyway, because the class said to fetch them every time. On a model with three
relationships, all set that way, every query about it loads three more tables.

That makes `lazy="selectin"` on the class an answer to the wrong question. It is the right one in a
small number of places, where a relationship is genuinely part of the thing and is wanted every
time, and the way to tell is that leaving it off would mean writing the same option on every query.
Otherwise the option belongs on the query.

### Loading for what happens after the session closes

There is a second reason to load a relationship on purpose, and it has nothing to do with counting.
A relationship that was never loaded cannot be loaded once the session is gone:


In [8]:
with Session(engine) as session:
    loaded = session.exec(select(Hero).options(joinedload(Hero.team)).limit(2)).all()
    plain = session.exec(select(Hero).offset(2).limit(2)).all()

print("loaded on purpose:", [(hero.name, hero.team.name) for hero in loaded])
try:
    print("not loaded       :", [(hero.name, hero.team.name) for hero in plain])
except DetachedInstanceError as error:                              # its message names a memory address
    print("not loaded       :", type(error).__name__ + ":", message(error).split(";")[0])


loaded on purpose: [('Iron-Fox', 'Iron Squad'), ('Iron-Hawk', 'Silver Squad')]
not loaded       : DetachedInstanceError: Parent instance <Hero at 0x...> is not bound to a Session


The same two lines, and only one of them works. This is what makes the option necessary rather than
merely faster in a service, where the session belongs to the request and everything read afterwards
has to have been loaded while it was open.

### A roster in two queries, finished

The pieces of this notebook in one function. `roster` is a list page: every team with its heroes,
loaded on purpose, counted so that the cost is a fact rather than a hope:


In [9]:
def roster(session, page=1, per_page=5):
    """One page of teams with their heroes, in two statements however many teams there are."""
    listed = (select(Team).order_by(Team.name)
              .offset((page - 1) * per_page).limit(per_page)
              .options(selectinload(Team.heroes)))
    return [{"team": team.name, "heroes": len(team.heroes),
             "youngest": min(hero.age for hero in team.heroes)}
            for team in session.exec(listed).all()]


with Session(engine) as session, counting(engine) as sent:
    page = roster(session)
for row in page[:3]:
    print(" ", row)
print("statements for the page:", dict(sent))

with Session(engine) as session, counting(engine) as sent:
    wider = roster(session, per_page=20)
print("and for twenty teams   :", dict(sent), "| rows:", len(wider))


  {'team': 'Bear Watch', 'heroes': 10, 'youngest': 39}
  {'team': 'Cobalt Squad', 'heroes': 10, 'youngest': 28}
  {'team': 'Crane Watch', 'heroes': 10, 'youngest': 36}
statements for the page: {'SELECT': 2}
and for twenty teams   : {'SELECT': 2} | rows: 20


Five teams in two statements, and twenty teams in two statements. The `youngest` line reads every
hero of every team, which would have been twenty more queries without the option and is free with
it, since the heroes are already in memory.

### Where each part came from

| In `roster` | What it relies on | The section that showed it |
|---|---|---|
| `.options(selectinload(Team.heroes))` | one extra statement for the whole page | `selectinload`: one more query for the whole page |
| `.offset(...).limit(...)` | a page in a fixed order | the **Reading Rows** notebook |
| `len(team.heroes)` and `min(...)` sending nothing | a relationship already in memory | Counting what a loop sends |
| `counting(engine)` | statements counted rather than timed | Setup |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/10-loading-and-n-plus-one-solutions.ipynb).

**1.** Count the statements a loop sends that prints the name of the team of every hero over 50, and
say in a comment why the number is what it is.


In [10]:
# your code here


**2.** Do the same with `joinedload`, and print both counts beside each other.


In [11]:
# your code here


**3.** Print the ten oldest heroes with their team names in as few statements as you can, and count
them.


In [12]:
# your code here


**4.** Count the statements for a page of three teams with their heroes, without any option and with
`selectinload`, and print the two counts.


In [13]:
# your code here


**5.** Load two teams with their heroes so that the names of both teams and all their heroes can be
printed after the session block has ended.


In [14]:
# your code here


**6.** Write `team_sizes(session)`, returning a dictionary of team name to hero count, in exactly one
statement. `func.count` and `group_by` are the way, and no relationship is needed.


In [15]:
# your code here


## Common errors

### No error, and twenty-one statements for one page


In [16]:
def slow_page(session):
    """Every team with how many heroes it has, and one query for each of them."""
    return [(team.name, len(team.heroes)) for team in session.exec(select(Team)).all()]


with Session(engine) as session, counting(engine) as sent:
    slow_page(session)
print("without an option:", dict(sent))

with Session(engine) as session, counting(engine) as sent:
    [(team.name, len(team.heroes))
     for team in session.exec(select(Team).options(selectinload(Team.heroes))).all()]
print("with one         :", dict(sent))


without an option: {'SELECT': 21}
with one         : {'SELECT': 2}


The two functions return the same rows. One of them sends twenty-one statements and the other two,
and the difference is one `options` call. This is the error that never raises: the code is right,
the answer is right, and the cost is nineteen round trips that nobody asked for.

Finding it is the same in any program: count the statements around the code that builds a page, and
look for a number that grows with the rows.

### sqlalchemy.orm.exc.DetachedInstanceError: Parent instance <Hero at 0x...> is not bound to a Session; lazy load operation of attribute 'team' cannot proceed


In [17]:
def heroes_for_the_page(engine, how_many):
    """A few heroes, read in a session of its own."""
    with Session(engine) as session:
        return session.exec(select(Hero).limit(how_many)).all()


heroes = heroes_for_the_page(engine, 2)
try:
    print([(hero.name, hero.team.name) for hero in heroes])
except DetachedInstanceError as error:
    print(type(error).__name__ + ":", message(error).split(";")[0])


DetachedInstanceError: Parent instance <Hero at 0x...> is not bound to a Session


The heroes' own columns are loaded and readable; the team was never fetched, and the session that
would have fetched it has gone. In a service this is the shape it arrives in: a function reads rows
in a session of its own, a template or a response model reads a relationship off them afterwards,
and the failure is in code that has nothing wrong with it.

Load what the caller will need, where the session still exists:


In [18]:
def heroes_with_teams(engine, how_many):
    """The same heroes, with their teams already loaded."""
    with Session(engine) as session:
        return session.exec(select(Hero).options(joinedload(Hero.team)).limit(how_many)).all()


print([(hero.name, hero.team.name) for hero in heroes_with_teams(engine, 2)])


[('Iron-Fox', 'Iron Squad'), ('Iron-Hawk', 'Silver Squad')]


### No error, and two queries for one column: a strategy set on the class


In [19]:
run_python("scratch/eager_model.py")


a query that needs only names: {'SELECT': 2} and it loaded 20 teams


The same file as the worked example, and worth reading twice as an error rather than as a feature: a
query that asked for team names sent a second statement for heroes nobody wanted. It is the
over-correction that follows the first error in this section, and it is hard to see afterwards,
because the code that pays for it does not mention the relationship at all.

The rule that avoids both is the same one: decide per query. `selectinload` where a page needs the
other side, nothing where it does not, and `lazy="selectin"` on the class only where every query
that touches the model wants it.

Last, the engine lets go of the file, and this cell removes the scratch folder with the database and
the script in it:


In [20]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- A relationship read in a loop sends one query per distinct object on the other side, which is the
  N+1: twenty teams, twenty-one statements.
- Count statements rather than time them: the count is the same on every machine, and it is what
  grows with the size of a page.
- `selectinload` fetches the relationship for a whole page in one more statement; `joinedload`
  fetches it in the same statement, with a join.
- `joinedload` suits a many-to-one, `selectinload` a one-to-many or a many-to-many, and either is
  better than a loop.
- A strategy on the class applies to every query, wanted or not; the option belongs on the query,
  where it can be decided page by page.


## What is next

The **sa_column and __table_args__** notebook is where `Field` runs out: a column that needs a real
SQLAlchemy type, a constraint over two columns, a check the database enforces, and the rule that
`sa_column` replaces what `Field` would have said rather than adding to it.


---

&#8592; **Previous:** [Many to Many](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/09-many-to-many.ipynb)  &nbsp;·&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [sa_column and __table_args__](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/11-sa-column-and-table-args.ipynb) &#8594;
